In [ ]:
import torch
from torch import nn
device = "cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
import zipfile
import requests
from pathlib import Path

# Set path for the folder
data_path = Path("data/")
image_path = data_path/"pizza_steak_sushi"
# If the image folder doesn't exist, download it and prepare it..
if image_path.is_dir():
  print(f"{image_path} directory exist")
else:
  print(f"{image_path} directory does not exist making new one...")
  image_path.mkdir(parents=True,
                   exist_ok=True)
# dwonloading zip data
with open(data_path / "pizza_steak_sushi.zip", "wb") as f:
  request = requests.get("https://github.com/AvisarBhandari/Ml/raw/main/Custom%20data/data/pizza_steak_sushi_20_percent.zip")
  print("Downloading data ...")
  f.write(request.content)
# Unzipping data
with zipfile.ZipFile(data_path /"pizza_steak_sushi.zip","r") as zip_ref:
  print("Unzipping data ...")
  zip_ref.extractall(image_path)

In [ ]:
import os
def walk_through_dir(dir_path):
  """Walks through dir_path returning its contents."""
  for dirpath, dirnames, filenames in os.walk(dir_path):
    print(f"There are {len(dirnames)} directories and {len(filenames)} images in '{dirpath}'.")

In [ ]:
walk_through_dir(image_path)

In [ ]:
train_dir = image_path / "train"
test_dir = image_path / "test"
train_dir , test_dir

In [ ]:
import random
from PIL import Image

# 1 Get Image Path
image_path_list = list(image_path.glob("*/*/*.jpg"))
# 2 pick random image
random_image_path = random.choice(image_path_list)
# 3 Get image class from path name (the image class is the name of the directory where the image is stored)
image_class = random_image_path.parent.stem
# 4 open the image
img = Image.open(random_image_path)
# 5. Print metadata
print(f"Random image path: {random_image_path}")
print(f"Image class: {image_class}")
print(f"Image height: {img.height}")
print(f"Image width: {img.width}")
img

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
image_as_array = np.asarray(img)
plt.figure(figsize=(10,7))
plt.title(f"Image Class: {image_class} | Image shape: {image_as_array.shape} -> (hight, width,color channel) [HWC]")
plt.imshow(image_as_array)
plt.axis(False)

## Transforming data


In [ ]:
import torch
from torch.utils.data import dataloader
from torchvision import transforms , datasets
# transform for image
data_transfor = transforms.Compose([
    # resize image
    transforms.Resize(size=(64,64)),
    # flip the image randomly on the horizontal
    transforms.RandomHorizontalFlip(p=0.5),
    # transfro to tensor
    transforms.ToTensor()
])

In [ ]:
data_transfor(img).shape

In [ ]:
def plot_transformed_image(image_path:list, transform, n=3, seed = None):
  """
  Selects random images from a path of images and loads/transforms
  them then plots the original vs the transformed version.
  """
  if seed:
    random.seed(seed)
  random_image_paths = random.sample(image_path,k=n)
  for image_path in random_image_paths:
    with Image.open(image_path) as f:
      fig, ax = plt.subplots(nrows=1, ncols=2)
      ax[0].imshow(f)
      ax[0].set_title(f"Orignal \n size: {f.size}")
      ax[0].axis(False)
      # Transform and plot target image
      transform_image = transform(f).permute(1,2,0) # change shape for matplotlib (C, H, W) -> (H, W, C)
      ax[1].imshow(transform_image)
      ax[1].set_title(f"Transform \n Shape: {transform_image.shape}")
      ax[1].axis(False)

      fig.suptitle(f"Class: {image_path.parent.stem}")

plot_transformed_image(image_path=image_path_list, transform=data_transfor,seed=42)

## Option 1: Loading image data using `ImageFolder`


In [ ]:
from torchvision import datasets
train_data = datasets.ImageFolder(root=train_dir,
                                  transform=data_transfor, # transform the image
                                  target_transform=None # transfrom the label
                                  )
test_data = datasets.ImageFolder(root=test_dir,transform=data_transfor)
train_data , test_data

In [ ]:
train_dir,test_dir

In [ ]:
# Get class name
class_name = train_data.classes
class_name

In [ ]:
# Get class names as dict
class_dict = train_data.class_to_idx
class_dict

In [ ]:
len(train_data), len(test_data)

In [ ]:
test_data.samples[0], test_data.targets[0]

In [ ]:
# Index on the train_data Dataset to get a single image and label
img, label = train_data[0][0], train_data[0][1]
print(f"Image tensor:\n{img}")
print(f"Image shape: {img.shape}")
print(f"Image datatype: {img.dtype}")
print(f"Image label: {label}")
print(f"Label datatype: {type(label)}")

In [ ]:
# Rearrange the order of dimensions
img_permute = img.permute(1, 2, 0)

# Print out different shapes (before and after permute)
print(f"Original shape: {img.shape} -> [color_channels, height, width]")
print(f"Image permute shape: {img_permute.shape} -> [height, width, color_channels]")

# Plot the image
plt.figure(figsize=(10, 7))
plt.imshow(img.permute(1, 2, 0))
plt.axis("off")
plt.title(class_name[label], fontsize=14);

In [ ]:
import os
os.cpu_count()

Turn loaded images into `DataLoader`'s

In [ ]:
from torch.utils.data import DataLoader
BATCH_SIZE = 1
train_dataloader = DataLoader(dataset=train_data,
                              batch_size=BATCH_SIZE,
                              num_workers=1, # how many subprocesses to use for data loading? (higher = more)
                              shuffle=True
                              )
test_dataloader = DataLoader(dataset=test_data,
                             batch_size=BATCH_SIZE,
                             num_workers=1,
                             shuffle=True)
test_dataloader,train_dataloader

In [ ]:
len(train_dataloader) , len(test_dataloader)

In [ ]:
img , label  = next(iter(train_dataloader))
print(f"Image Shape: {img.shape} -> [batch_size, color_channelsr, hight, width]")
print(f"Label Shaep: {label.shape}")

### Option 2: Loading Image Data with a Custom `Dataset`

In [ ]:
import os
import pathlib
import torch

from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms
from typing import Tuple, Dict, List


In [ ]:
# Instance of torchvision.datasets.ImageFolder()
train_data.classes, train_data.class_to_idx

In [ ]:
# setup target directory
target_directory = train_dir
print(f"Target dir: {target_directory}")

# Get class name from target directory
class_names_found = sorted([entry.name for entry in list(os.scandir(target_directory))])
class_names_found

In [ ]:
list(os.scandir(target_directory)) , os.scandir(target_directory)

In [ ]:
def find_classes(directory:str) -> Tuple[List[int],Dict[str,int]]:
  """ Find Class folder name in the target directory """
  # Get the class name by scanning the target directory
  classes = sorted(entry.name for entry in os.scandir(directory) if entry.is_dir)

  # error if class name not found
  if not classes:
    raise FileNotFoundError(f"File not find in {directory}")

  # dictionary if index label
  class_to_idx = {class_name:i for i ,class_name in enumerate(classes)}

  return classes , class_to_idx

In [ ]:
find_classes(target_directory)

## Create a custom `Dataset` to replicate `ImageFolder`

To create our own custom dataset, we want to:

1. Subclass `torch.utils.data.Dataset`
2. Init our subclass with a target directory (the directory we'd like to get data from) as well as a transform if we'd like to transform our data.
3. Create several attributes:
  * paths - paths of our images
  * transform - the transform we'd like to use
  * classes - a list of the target classes
  * class_to_idx - a dict of the target classes mapped to integer labels
4. Create a function to `load_images()`, this function will open an image
5. Overwrite the `__len__()` method to return the length of our dataset
6. Overwrite the `__getitem__()` method to return a given sample when passed an index

In [ ]:
# make custom dataset class
from torch.utils.data import Dataset
# subclass "torch.utils.data.dataset"
class ImageFolderCustom(Dataset):
  # initialize our custom dataset
  def __init__(self , targ_dir:str , transform=None):
    # Get all image path
    self.paths = list(pathlib.Path(targ_dir).glob("*/*.jpg"))
    # setup transform
    self.transform = transform
    # classes and class_to_idx attributes
    self.classes , self.class_to_idx = find_classes(targ_dir)

    # function to load image
  def load_image(self , index: int) -> Image.Image:
    """ Open image via path and return it """
    image_path = self.paths[index]
    return Image.open(image_path)
      # operwrite __len__()
  def __len__(self) -> int:
    return len(self.paths)
  def __getitem__(self,index:int) -> Tuple[torch.Tensor , int]:
    img = self.load_image(index)
    class_name = self.paths[index].parent.name
    class_idx = self.class_to_idx[class_name]

    # transform if necessey
    if self.transform:
      return self.transform(img) , class_idx
    else:
      return img , class_idx


In [ ]:
# create transform
from torchvision import transforms
train_transform = transforms.Compose([
    transforms.Resize(size=(64,64)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor()
])
test_transfort = transforms.Compose([
    transforms.Resize(size=(64,64)),
    transforms.ToTensor()
])

In [ ]:
# test custome ImageFolderCustom
train_data_custom = ImageFolderCustom(targ_dir=train_dir,transform=train_transform)
test_data_custom = ImageFolderCustom(targ_dir=test_dir,transform=test_transfort)

In [ ]:
train_data_custom, test_data_custom

In [ ]:
len(train_data), len(train_data_custom)

In [ ]:
len(test_data), len(test_data_custom)

In [ ]:
train_data_custom.classes

In [ ]:
train_data_custom.class_to_idx

In [ ]:
train_data_custom[1]

In [ ]:
# Quility between ImageFolder and custom ImageFolder
print(train_data_custom.classes == train_data.classes)
print(test_data_custom.classes == test_data.classes)

In [ ]:
# function to display random dataset
def display_random_image(dataset:torch.utils.data.Dataset,classes:list[str],n:int = 10, display_shape:bool = True , seed:int =None):
  # Adjust display if n is too high
  if n > 10:
    n = 10
    display_shape = False
    print(f"For display purposes n should not be grater the 10 , setting n as 10 and removing shape display")
    # set seed
  if seed:
    random.seed(seed)
  # Get random sample
  random_sample_idx = random.sample(range(len(dataset)) , k=n)
  # setup plot
  plt.figure(figsize=(16,8))
  # loop through and plot image
  for i, targ_sample in enumerate(random_sample_idx):
    targ_image , targ_label = dataset[targ_sample][0], dataset[targ_sample][1]
    # Adjust tensor dimensions for plotting
    targ_image_adjust = targ_image.permute(1,2,0) # [color_channels, height, width] -> [height, width, color_channels]
    # Plot adjusted samples
    if i<5:
      plt.subplot(1,n,i+1)
    else:
      plt.subplot(2,n,i-4)

    plt.imshow(targ_image_adjust)
    plt.axis("off")
    if classes:
      title = f"Classes: {classes[targ_label]}"
      if display_shape:
        title = title+ f"\n Shape: {targ_image_adjust.shape}"
    plt.title(title)



In [ ]:
# Display random images from the ImageFolder created Dataset
display_random_image(dataset=train_data_custom,classes=class_name,n=5 )

In [ ]:
display_random_image(dataset=train_data_custom,classes=class_name,n=50 )

In [ ]:
from torch.utils.data import DataLoader
BATCH_SIZE = 32
NUM_WORKERS = os.cpu_count()
train_dataloader_custom = DataLoader(dataset=train_data_custom,
                                     batch_size=BATCH_SIZE,
                                     shuffle=True,
                                     num_workers=NUM_WORKERS)
test_dataloader_custom = DataLoader(dataset=test_data_custom,
                                    batch_size=BATCH_SIZE,
                                    shuffle=False,
                                    num_workers=NUM_WORKERS)
train_dataloader_custom, test_dataloader_custom

In [ ]:
img_custom, label_custom = next(iter(train_dataloader_custom))

img_custom.shape, label_custom.shape

###  Other forms of transforms (data augmentation)

In [ ]:
# https://docs.pytorch.org/vision/stable/auto_examples/transforms/plot_transforms_illustrations.html#trivialaugmentwide
from torchvision import transforms
train_transform = transforms.Compose([
    transforms.Resize(size=(224,225)),
    transforms.TrivialAugmentWide(num_magnitude_bins=31),
    transforms.ToTensor()
])
test_transfort = transforms.Compose([
    transforms.Resize(size=(224,224)),
    transforms.ToTensor()
])

In [ ]:
image_path

In [ ]:
image_path_list = list(image_path.glob("*/*/*.jpg"))
image_path_list[:5]

In [ ]:
plot_transformed_image(
    image_path=image_path_list,
    transform=train_transform,
    n=3,
    seed=None
)

###  Model 0: TinyVGG without data augmentation

In [ ]:
simple_transform = transforms.Compose([
    transforms.Resize(size=(64,64)),
    transforms.ToTensor()
])

In [ ]:
# load and transform data
from torchvision import datasets
train_simple = datasets.ImageFolder(root=train_dir,transform=simple_transform)
test_simple = datasets.ImageFolder(root=test_dir, transform=simple_transform)
# turn the data into dataloader
import os
from torch.utils.data import DataLoader
BATCH_SIZE = 32
NUM_WORKERS = os.cpu_count()
train_dataloader_simple = DataLoader(dataset=train_simple , batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
test_dataloader_simple = DataLoader(dataset=test_simple , batch_size=BATCH_SIZE , num_workers=NUM_WORKERS)

In [ ]:
NUM_WORKERS

In [ ]:
class TinyVGG(nn.Module):
  """Model architecture copying TinyVGG from CNN Explainer: https://poloclub.github.io/cnn-explainer/"""
  def __init__(self, input_shape:int, output_shape:int, hidden_units:int) -> None:
    super().__init__()
    self.conv_block_1 = nn.Sequential(
        nn.Conv2d(in_channels=input_shape, out_channels=hidden_units,stride=1,padding=1,kernel_size=3),
        nn.ReLU(),
        nn.Conv2d(in_channels=hidden_units,out_channels=hidden_units, stride=1,padding=1,kernel_size=3),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2,stride=2)
    )
    self.conv_block_2 = nn.Sequential(
        nn.Conv2d(in_channels=hidden_units,out_channels=hidden_units,kernel_size=3,stride=1,padding=1),
        nn.ReLU(),
        nn.Conv2d(in_channels=hidden_units, out_channels=hidden_units, kernel_size=3, stride=1, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2)
    )
    self.classifier = nn.Sequential(
        nn.Flatten(),
        nn.Linear(in_features=hidden_units*16*16 , out_features=output_shape)
    )
  def forward(self,x:torch.tensor):
    x = self.conv_block_1(x)
    # print(x.shape)
    x = self.conv_block_2(x)
    # print(x.shape)
    x = self.classifier(x)
    # print(x.shape)
    return x
    # return self.classifier(self.conv_block_2(self.conv_block_1(x))) # <- leverage the benefits of operator fusion


torch.manual_seed(42)
model_0 = TinyVGG(input_shape=3 , hidden_units=10, output_shape=len(train_data.classes)).to(device)

In [ ]:
image_batch, label_batch = next(iter(train_dataloader_simple))
image_batch.shape, label_batch.shape

In [ ]:
model_0(image_batch)

In [ ]:
torch.randn(1,3,64,64).shape

### `torchinfo` to get an idea of the shapes going through our model

In [ ]:
# Install torchinfo, import if it's available
try:
  import torchinfo
except:
  !pip install torchinfo
  import torchinfo

from torchinfo import summary
summary(model_0, input_size=[1, 3, 64, 64])

In [ ]:
def train_step(model:torch.nn.Module, dataloader:torch.utils.data.DataLoader, loss_fn:torch.nn.Module, optimizer:torch.optim.Optimizer, device=device):
  model.train()
  train_acc , train_loss = 0, 0
  for batch, (X,y) in enumerate(dataloader):
    X , y = X.to(device) , y.to(device)
    y_pred = model(X)
    loss = loss_fn(y_pred, y)
    train_loss +=loss.item()
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
     # Calculate accuracy metric
    y_pred_class = torch.argmax(torch.softmax(y_pred , dim=1),dim=1)
    train_acc += (y_pred_class ==y).sum().item()/len(y_pred)
    # Adjust metrics to get average loss and accuracy per batch
  train_loss = train_loss / len(dataloader)
  train_acc = train_acc / len(dataloader)
  return train_loss , train_acc

In [ ]:
def test_step(model:torch.nn.Module, dataloader:torch.utils.data.DataLoader, loss_fn:torch.nn.Module, device = device):
  model.eval()
  test_loss, test_acc = 0,0
  with torch.inference_mode():
    for batch, (X,y) in enumerate(dataloader):
      X, y = X.to(device),y.to(device)
      test_logits = model(X)
      loss = loss_fn(test_logits , y)
      test_loss += loss.item()
      # Calculate the accuracy
      test_pred_labels = test_logits.argmax(dim=1)
      test_acc += ((test_pred_labels == y).sum().item() / len(test_pred_labels))
    # Adjust metrics to get average loss and accuracy per batch
  test_loss = test_loss / len(dataloader)
  test_acc = test_acc / len(dataloader)
  return test_loss , test_acc

In [ ]:
from tqdm.auto import tqdm
# 1. train function
def train(model: torch.nn.Module,
          train_dataloader,
          test_dataloader,
          optimizer,
          loss_fn: torch.nn.Module = nn.CrossEntropyLoss(),
          epochs: int = 5,
          device=device):

  # 2. Create empty results dictionary
  results = {"train_loss": [],
             "train_acc": [],
             "test_loss": [],
             "test_acc": []}

  # 3. Loop through training and testing steps for a number of epochs
  for epoch in tqdm(range(epochs)):
    train_loss, train_acc = train_step(model=model,
                                       dataloader=train_dataloader,
                                       loss_fn=loss_fn,
                                       optimizer=optimizer,
                                       device=device)
    test_loss, test_acc = test_step(model=model,
                                    dataloader=test_dataloader,
                                    loss_fn=loss_fn,
                                    device=device)

    # 4. Print out what's happening
    print(f"Epoch: {epoch} | Train loss: {train_loss:.4f} | Train acc: {train_acc:.4f} | Test loss: {test_loss:.4f} | Test acc: {test_acc:.4f}")

    # 5. Update results dictionary
    results["train_loss"].append(train_loss)
    results["train_acc"].append(train_acc)
    results["test_loss"].append(test_loss)
    results["test_acc"].append(test_acc)

  # 6. Return the filled results at the end of the epochs
  return results

In [ ]:
# Set random seeds
torch.manual_seed(42)
torch.cuda.manual_seed(42)

# Set number of epochs
NUM_EPOCHS = 5

# Recreate an instance of TinyVGG
model_0 = TinyVGG(input_shape=3, # number of color channels of our target images
                  hidden_units=10,
                  output_shape=len(train_data.classes)).to(device)

# Setup loss function and optimizer
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=model_0.parameters(),
                             lr=0.001)

# Start the timer
from timeit import default_timer as timer
start_time = timer()

# Train model_0
model_0_results = train(model=model_0,
                        train_dataloader=train_dataloader_simple,
                        test_dataloader=test_dataloader_simple,
                        optimizer=optimizer,
                        loss_fn=loss_fn,
                        epochs=NUM_EPOCHS)

# End the timer and print out how long it took
end_time = timer()
print(f"Total training time: {end_time-start_time:.3f} seconds")

In [ ]:
model_0_results

In [ ]:
model_0_results.keys()

In [ ]:
def plot_loss_curves(results: Dict[str, List[float]]):
    """Plots training curves of a results dictionary.

    Args:
        results (dict): dictionary containing list of values, e.g.
            {"train_loss": [...],
             "train_acc": [...],
             "test_loss": [...],
             "test_acc": [...]}
    """

    # Get the loss values of the results dictionary (training and test)
    loss = results['train_loss']
    test_loss = results['test_loss']

    # Get the accuracy values of the results dictionary (training and test)
    accuracy = results['train_acc']
    test_accuracy = results['test_acc']

    # Figure out how many epochs there were
    epochs = range(len(results['train_loss']))

    # Setup a plot
    plt.figure(figsize=(15, 7))

    # Plot loss
    plt.subplot(1, 2, 1)
    plt.plot(epochs, loss, label='train_loss')
    plt.plot(epochs, test_loss, label='test_loss')
    plt.title('Loss')
    plt.xlabel('Epochs')
    plt.legend()

    # Plot accuracy
    plt.subplot(1, 2, 2)
    plt.plot(epochs, accuracy, label='train_accuracy')
    plt.plot(epochs, test_accuracy, label='test_accuracy')
    plt.title('Accuracy')
    plt.xlabel('Epochs')
    plt.legend();

In [ ]:
plot_loss_curves(results=model_0_results)

### Compare model results

After evaluating our modelling experiments on their own, it's important to compare them to each other.

There's a few different ways to do this:
1. Hard coding (what we're doing)
2. PyTorch + Tensorboard - https://pytorch.org/docs/stable/tensorboard.html
3. Weights & Biases - https://wandb.ai/site/experiment-tracking
4. MLFlow - https://mlflow.org/


### Model 1: TinyVGG with Data Augmentation

In [ ]:
# Create training transform with TriviailAugment
from torchvision import transforms
train_transform_trivial = transforms.Compose([
    transforms.Resize(size=(64,64)),
    transforms.TrivialAugmentWide(num_magnitude_bins=31),
    transforms.ToTensor()
])
test_transfort_simple = transforms.Compose([
    transforms.Resize(size=(64,64)),
    transforms.ToTensor()
])

### train and test `Dataset`'s and `DataLoader`'s with data augmentation

In [ ]:
from torchvision import datasets
train_data_augmentated = datasets.ImageFolder(root=train_dir,transform=train_transform_trivial)
test_data_simple = datasets.ImageFolder(root=test_dir,transform=test_transfort_simple)

In [ ]:
import os
from torch.utils.data import DataLoader
BATCH_SIZE = 32
NUM_WORKERS = os.cpu_count()
torch.manual_seed(42)
train_dataloader_augmentated = DataLoader(dataset=train_data_augmentated,batch_size =BATCH_SIZE, shuffle=True,num_workers=NUM_WORKERS)
test_dataloader_simple = DataLoader(dataset=test_data_simple, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

### Construct and train model 1

This time we'll be using the same model architecture except this time we've augmented the training data.

In [ ]:
model_1 = TinyVGG(input_shape=3, output_shape=len(train_data_augmentated.classes), hidden_units=10).to(device)
model_1

In [ ]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)

EPOCHS = 5

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=model_1.parameters(),lr=0.001)

from timeit import default_timer as timer
start_time = timer()
model_1_result = train(model=model_1,
                       train_dataloader=train_dataloader_augmentated,
                       test_dataloader=test_dataloader_simple,
                       optimizer=optimizer,
                       loss_fn=loss_fn,
                       epochs=EPOCHS)
end_time = timer()
print(f"Total traning time for model_1:{end_time-start_time:.3f} secound")

In [ ]:
model_1_result

In [ ]:
plot_loss_curves(model_1_result)

### Compare model results

After evaluating our modelling experiments on their own, it's important to compare them to each other.

There's a few different ways to do this:
1. Hard coding (what we're doing)
2. PyTorch + Tensorboard - https://pytorch.org/docs/stable/tensorboard.html
3. Weights & Biases - https://wandb.ai/site/experiment-tracking
4. MLFlow - https://mlflow.org/


In [ ]:
import pandas as pd
model_0_df = pd.DataFrame(model_0_results)
model_1_df = pd.DataFrame(model_1_result)
model_0_df

In [ ]:
plt.figure(figsize=(15,10))

epochs = range(len(model_1_df))

# plot train loss
plt.subplot(2,2,1)
plt.plot(epochs,model_0_df["train_loss"],label="model 0")
plt.plot(epochs,model_1_df["train_loss"],label="model 1")
plt.title("Train Loss")
plt.xlabel("Epochs")
plt.legend()
# plot test loss
plt.subplot(2,2,2)
plt.plot(epochs,model_0_df["test_loss"],label="model 0")
plt.plot(epochs,model_1_df["test_loss"],label="model 1")
plt.title("test Loss")
plt.xlabel("Epochs")
plt.legend()
# plot train accuracy
plt.subplot(2,2,3)
plt.plot(epochs,model_0_df["train_acc"],label="model 0")
plt.plot(epochs,model_1_df["train_acc"],label="model 1")
plt.title("Train Accuracy")
plt.xlabel("Epochs")
plt.legend()
# plot test accuracy
plt.subplot(2,2,4)
plt.plot(epochs,model_0_df["test_acc"],label="model 0")
plt.plot(epochs,model_1_df["test_acc"],label="model 1")
plt.title("Test Accuracy")
plt.xlabel("Epochs")
plt.legend()

In [ ]:
# 1. Create a function to take in a dataset
def display_random_pred(model: torch.nn.Module,
                        dataset: torch.utils.data.Dataset,
                        n: int = 5,
                        seed: int = None,
                          ):
  # 2. Adjust display if n is too high
  if n > 5:
    n = 5
    print(f"For display, purposes, n shouldn't be larger than 10, setting to 10 and removing shape display.")

  # 3. Set the seed
  if seed:
    random.seed(seed)

  # 4. Get random sample indexes and get classes
  random_samples_idx = random.sample(range(len(dataset)), k=n)
  classes = dataset.classes
  # 5. Setup plot
  plt.figure(figsize=(16, 8))

  # 6. Loop through random indexes and plot them with matplotlib
  for i, targ_sample in enumerate(random_samples_idx):
    targ_image, targ_label = dataset[targ_sample][0], dataset[targ_sample][1]
    with torch.inference_mode():
      logits = model(targ_image.unsqueeze(0))
      pred = logits.argmax(dim=1)


    # 7. Adjust tensor dimensions for plotting
    targ_image_adjust = targ_image.permute(1, 2, 0) # [color_channels, height, width] -> [height, width, color_channels]

    # Plot adjusted samples
    plt.subplot(1, n, i+1)
    plt.imshow(targ_image_adjust)
    plt.axis("off")
    title = f"Class: {classes[targ_label]}"
    title = title + f"\n pred: {classes[pred]}"
    text_color = 'green' if classes[targ_label] == classes[pred] else 'red'
    plt.title(title, color=text_color)

In [ ]:
display_random_pred(dataset=test_data_simple,model=model_0,seed=42), display_random_pred(dataset=test_data_simple,model=model_1,seed=42)

In [ ]:
# Download custom image
import requests

# Setup custom image path
custom_image_path = data_path / "test-pizza.jpeg"

# Download the image if it doesn't already exist
if not custom_image_path.is_file():
  with open(custom_image_path, "wb") as f:
    # When downloading from GitHub, need to use the "raw" file link
    request = requests.get("https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcTlzidb7kuobJidDMpRxP0Xtx_dQCnCz1yBwQ&s")
    print(f"Downloading {custom_image_path}...")
    f.write(request.content)
else:
  print(f"{custom_image_path} already exists, skipping download...")

In [ ]:
custom_image_path

In [ ]:
import torchvision

# Read in custom image
custom_image_uint8 = torchvision.io.decode_image(str(custom_image_path))
print(f"Custom image tensor:\n {custom_image_uint8}")
print(f"Custom image shape: {custom_image_uint8.shape}")
print(f"Custom image datatype: {custom_image_uint8.dtype}")

In [ ]:
plt.imshow(custom_image_uint8.permute(1,2,0))

In [ ]:
custom_image = custom_image_uint8.type(torch.float32) / 255.
print(f"Custom image tensor:\n {custom_image}")
print(f"Custom image shape: {custom_image.shape}")
print(f"Custom image datatype: {custom_image.dtype}")

In [ ]:
# Create transform pipeline to resize image
from torchvision import transforms
custom_image_transform = transforms.Compose([
                                             transforms.Resize(size=(64, 64))
])

# Transfrom target image
custom_image_transformed = custom_image_transform(custom_image)

# Print out the shapes
print(f"Original shape: {custom_image.shape}")
print(f"Transformed shape: {custom_image_transformed.shape}")

In [ ]:
plt.imshow(custom_image_transformed.permute(1, 2, 0))

In [ ]:
# This should this work? (added a batch size...)
model_1.eval()
with torch.inference_mode():
  custom_image_pred = model_1(custom_image_transformed.unsqueeze(0).to(device))
custom_image_pred

In [ ]:
# Convert logits -> prediction probabilities
custom_image_pred_probs = torch.softmax(custom_image_pred, dim=1)
custom_image_pred_probs

In [ ]:
# Convert prediction probabilities -> prediction labels
custom_image_pred_label = torch.argmax(custom_image_pred_probs, dim=1).cpu()
custom_image_pred_label

In [ ]:
class_name[custom_image_pred_label]

 ### Putting custom image prediction together: building a function

In [ ]:
def pred_and_plot_image(model: torch.nn.Module,
                        image_path: str,
                        class_names: List[str] = None,
                        transform=None,
                        device=device):
  """Makes a prediction on a target image with a trained model and plots the image and prediction."""
  # Load in the image
  target_image = torchvision.io.decode_image(str(image_path)).type(torch.float32)

  # Divide the image pixel values by 255 to get them between [0, 1]
  target_image = target_image / 255.

  # Transform if necessary
  if transform:
    target_image = transform(target_image)

  # Make sure the model is on the target device
  model.to(device)

  # Turn on eval/inference mode and make a prediction
  model.eval()
  with torch.inference_mode():
    # Add an extra dimension to the image (batch dimension, e.g. our model will predict on batches of 1x image)
    target_image = target_image.unsqueeze(0)

    # Make a prediction on the image with an extra dimension
    target_image_pred = model(target_image.to(device))

  # logits -> prediction probabilities
  target_image_pred_probs = torch.softmax(target_image_pred, dim=1)

  # predction probabilities -> prediction labels
  target_image_pred_label = torch.argmax(target_image_pred_probs, dim=1)

  # Plot the image alongside the prediction and prediction probability
  plt.imshow(target_image.squeeze().permute(1, 2, 0)) # remove batch dimension and rearrange shape to be HWC
  if class_names:
    title = f"Pred: {class_names[target_image_pred_label.cpu()]} | Prob: {target_image_pred_probs.max().cpu():.3f}"
  else:
    title = f"Pred: {target_image_pred_label} | Prob: {target_image_pred_probs.max().cpu():.3f}"
  plt.title(title)
  plt.axis(False)

In [ ]:
# Pred on our custom image
pred_and_plot_image(model=model_1,
                    image_path=custom_image_path,
                    class_names=class_name,
                    transform=custom_image_transform,
                    device=device)